In [11]:
# Fix Python path to include user-installed packages for BOTH Python 3.10 and 3.12
import sys
import site

# Add both Python 3.10 and 3.12 user site-packages
paths_to_add = [
    '/glade/u/home/wukoutian/.local/lib/python3.10/site-packages',
    '/glade/u/home/wukoutian/.local/lib/python3.12/site-packages',
    site.getusersitepackages()
]

for path in paths_to_add:
    if path not in sys.path:
        sys.path.insert(0, path)
        print(f"Added to path: {path}")

print("\nPython path updated. Now you can import ee and geemap.")
print(f"Current Python version: {sys.version}")


Python path updated. Now you can import ee and geemap.
Current Python version: 3.10.13 | packaged by conda-forge | (main, Oct 26 2023, 18:07:37) [GCC 12.3.0]


# Note: NumPy Compatibility Warning

If you see a NumPy 1.x vs 2.x warning, you can safely ignore it. The code works fine despite the warning.

To fix it permanently, you would need to downgrade NumPy, but this requires disk space:
```python
# pip install "numpy<2" --user
```

**Disk quota is currently full, so skip this step.**

In [12]:
import sys
import importlib

In [13]:
# Import Earth Engine
import ee
import geemap

# Authenticate (only needed first time)
ee.Authenticate()

# Initialize Earth Engine
ee.Initialize()

print("Earth Engine initialized successfully!")

Earth Engine initialized successfully!


In [14]:
# Set up Earth Engine authentication
ee.Authenticate()
ee.Initialize()

# Load collection
dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

# Point of interest
point = ee.Geometry.Point(-121.8036, 39.0372)

# Get embedding images for two years
image1 = dataset.filterDate("2023-01-01", "2024-01-01").filterBounds(point).first()

image2 = dataset.filterDate("2024-01-01", "2025-01-01").filterBounds(point).first()

# Visualization parameters
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

# Calculate dot product (similarity measure)
dot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())


# Print out some information about the images
def print_image_details():
    print("<b>2023 Image Details:</b>")
    print(f"Bands: {image1.bandNames().getInfo()}")
    print(f"Image Projection: {image1.projection().getInfo()}")

    print("\n<b>2024 Image Details:</b>")
    print(f"Bands: {image2.bandNames().getInfo()}")
    print(f"Image Projection: {image2.projection().getInfo()}")

    print("\n<b>Dot Product Similarity:</b>")
    print(f"Similarity Value: {dot_prod.getInfo()}")


# Run the detailed analysis
print_image_details()

# Optional: Export images
# Uncomment the following line if you want to export images
# export_images()

<b>2023 Image Details:</b>
Bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']
Image Projection: {'type': 'Projection', 'crs': 'EPSG:32610', 'transform': [10, 0, 500000, 0, 10, 4259840]}

<b>2024 Image Details:</b>
Bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A

In [ ]:
# ============================================
# AlphaEarth Bitcoin Mining Detection Starter
# ============================================

import ee
import geemap

# 1. LOAD ALPHAEARTH EMBEDDINGS
alphaEarth = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

# 2. DEFINE YOUR ANALYSIS PERIOD
startYear = 2018  # Match paper's June 2018 start
endYear = 2024    # Current available data

# 3. LOAD MINING LOCATIONS FROM CSV
# Load the FeatureCollection from your GEE asset
miningLocations = ee.FeatureCollection('projects/ee-ktwu01/assets/bitcoin-mining')

# Convert CSV data to proper format
def format_mining_location(feature):
    lat = ee.Number(feature.get('Latitude'))
    lon = ee.Number(feature.get('Longitude'))
    country = feature.get('CRCode')
    countryName = feature.get('CRName')
    
    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'label': 1,
            'year': 2018,
            'location': country,
            'country_name': countryName
        }
    )

miningLocations = miningLocations.map(format_mining_location)

# Print number of mining locations
numPositive = miningLocations.size()
print('Number of mining locations:', numPositive.getInfo())

# 4. VISUALIZE ALL POSITIVE MINING LOCATIONS ON MAP
# Create an interactive map
Map = geemap.Map(zoom=2)
Map.centerObject(miningLocations, 2)  # Global view
Map.addLayer(miningLocations, {'color': 'red'}, 'Bitcoin Mining Locations')
Map

In [ ]:
# 5. GENERATE NEGATIVE SAMPLES (non-mining locations)
# Best practices for negative sampling:
# 1. Match the geographic distribution of positive samples
# 2. Include diverse land cover types (urban, rural, industrial, natural)
# 3. Use stratified random sampling within same regions
# 4. Aim for balanced dataset (1:1 or up to 1:3 positive:negative ratio)

# Get bounding box of mining locations to constrain negative sampling
miningBounds = miningLocations.geometry().bounds()

# Generate random points within the same geographic regions
negativeLocations = ee.FeatureCollection.randomPoints(
    region=miningBounds,
    points=numPositive,  # Match number of positive samples
    seed=42,  # For reproducibility
    maxError=1
)

# Add labels to negative samples
def add_negative_label(feature):
    return feature.set({
        'label': 0,
        'year': 2018,
        'location': 'negative_sample'
    })

negativeLocations = negativeLocations.map(add_negative_label)

# Optional: Filter out negative samples that are too close to mining sites
# This prevents contamination (e.g., excluding points within 1km of mining sites)
minDistance = 1000  # meters

def calculate_min_distance(negFeature):
    # Calculate distance to nearest mining location
    def calc_distance(posFeature):
        return negFeature.geometry().distance(posFeature.geometry())
    
    distances = miningLocations.map(calc_distance)
    minDist = distances.aggregate_min('distance')
    return negFeature.set('min_distance_to_mining', minDist)

negativeLocations = negativeLocations.map(calculate_min_distance)
negativeLocations = negativeLocations.filter(ee.Filter.gte('min_distance_to_mining', minDistance))

print('Number of negative samples after filtering:', negativeLocations.size().getInfo())

# 6. COMBINE POSITIVE AND NEGATIVE SAMPLES
trainingPoints = miningLocations.merge(negativeLocations)
print('Total training points:', trainingPoints.size().getInfo())

In [ ]:
# 9. EXPORT FOR CLASSIFIER TRAINING
# Generate band selectors for all 64 AlphaEarth bands
band_names = ['A' + str(i).zfill(2) for i in range(64)]
selectors = ['label', 'location', 'year'] + band_names

# Export 2018 data to Google Drive
task2018 = ee.batch.Export.table.toDrive(
    collection=embeddings2018,
    description='AlphaEarth_Mining_Training_2018',
    fileFormat='CSV',
    selectors=selectors
)

# Uncomment to start the export task
# task2018.start()
# print('Export task started. Check your Google Drive and Earth Engine Tasks tab.')

print('Export task configured. To start export, uncomment task2018.start() and run again.')
print(f'This will export {embeddings2018.size().getInfo()} samples with {len(band_names)} embedding bands.')

In [ ]:
# 10. VISUALIZE TRAINING POINTS ON MAP
# Create a comprehensive visualization
Map2 = geemap.Map(zoom=2)

# Get AlphaEarth image for 2018
ae2018 = alphaEarth.filter(ee.Filter.eq('year', 2018)).first()

# Add AlphaEarth as RGB (using bands A01, A16, A09)
vis_params = {
    'min': -0.3,
    'max': 0.3,
    'bands': ['A01', 'A16', 'A09']
}

Map2.addLayer(ae2018, vis_params, 'AlphaEarth 2018 RGB', False)
Map2.addLayer(miningLocations, {'color': 'FF0000'}, 'Mining Locations (Positive)')
Map2.addLayer(negativeLocations, {'color': '0000FF'}, 'Negative Samples')

# Center on first mining location
Map2.centerObject(miningLocations.first(), 12)

print('Training Points Summary:')
print('  Total:', trainingPoints.size().getInfo())
print('  Positive (Mining):', miningLocations.size().getInfo())
print('  Negative (Non-mining):', negativeLocations.size().getInfo())
print('\nAlphaEarth Bands:', ae2018.bandNames().getInfo())
print('\nMap Legend:')
print('  Red points = Bitcoin mining locations')
print('  Blue points = Negative samples (non-mining)')

Map2

In [ ]:
# 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR
def extractEmbeddings(year):
    """Extract AlphaEarth embeddings for a specific year"""
    yearImage = alphaEarth.filter(ee.Filter.eq('year', year)).first()
    
    # Sample the 64-band embeddings at each point
    samples = yearImage.sampleRegions(
        collection=trainingPoints,
        scale=10,  # AlphaEarth is 10m resolution
        geometries=True
    )
    
    return samples

# 8. EXTRACT FOR MULTIPLE YEARS
print('Extracting embeddings for multiple years...')
embeddings2018 = extractEmbeddings(2018)
embeddings2021 = extractEmbeddings(2021)  # Pre-China ban
embeddings2024 = extractEmbeddings(2024)  # Post-ban

print('Embeddings extracted successfully!')
print('2018 samples:', embeddings2018.size().getInfo())
print('2021 samples:', embeddings2021.size().getInfo())
print('2024 samples:', embeddings2024.size().getInfo())